# U2_T4 · Cuaderno 2 — Tu caso de estudio

**Mecatrónica · Métodos Numéricos**

Aquí se hace **todo el ejercicio**: se muestra tu caso, se aísla la raíz, se **eligen los
métodos**, se corre la **simulación de Monte Carlo**, se leen las gráficas y se califica la
selección.

> ### 👉 Antes de empezar
> - El **jefe de equipo** debe haber ejecutado `U2_T4_1_Equipo` (ahí se reserva el tema).
> - Cada integrante puede trabajar en **su propia copia** de este cuaderno.
> - **Puedes usar `Ejecutar todo`**: ninguna celda pide datos a mano ni reserva nada.

> 🎓 **La pregunta de la exposición:** *"Si estuvieran diseñando el firmware del controlador
> de este sistema mecatrónico en tiempo real, ¿qué método programarían y por qué?"*


In [ ]:
#@title ⚙️ Configuración general — ejecuta esta celda **PRIMERO** (no modificar)
# ============================================================
# Detecta si corre en Google Colab o en tu computadora, monta Drive y
# carga el motor de la tarea (ver MODO_MOTOR más abajo).
# ============================================================
import base64, importlib, os, subprocess, sys, zlib, json

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

# --- Repositorio de la tarea  <-- EDITAR SOLO SI CAMBIA EL REPO ------
REPO_URL = 'https://github.com/ITH-MGL-MN/U2_T4_Casos_de_estudio.git'
REPO_NOMBRE = 'U2_T4_Casos_de_estudio'

# --- Solo en tu computadora: de dónde sale el motor ---
#   "modulos" -> profe/** tal cual, importado como paquete (para depurar)
#   "bundle"  -> grader_local.py, el motor APLANADO (forma exacta de Colab)
#   "blob"    -> grader_ofuscado.txt, lo que recibe el alumno
MODO_MOTOR = "modulos"

if ES_COLAB:
    BASE = "/content/drive/MyDrive/CursoMN"
    REPO = os.path.join(BASE, REPO_NOMBRE)
else:
    REPO = os.getcwd()
    for _ in range(4):
        if os.path.exists(os.path.join(REPO, "grader_local.py")):
            break
        _padre = os.path.dirname(REPO)
        if _padre == REPO:
            break
        REPO = _padre
    if not os.path.exists(os.path.join(REPO, "grader_local.py")):
        REPO = r"o:\Mi unidad\CursoMN\U2_T4_Casos_de_estudio"

MODO = "blob" if ES_COLAB else MODO_MOTOR
print("Modo          :", "Google Colab" if ES_COLAB else "PC local")
print("Carpeta tarea :", REPO)

# ============================================================
# Montar Drive y traer el repositorio  (solo en Colab)
# ============================================================
if ES_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(BASE, exist_ok=True)
    try:
        if not os.path.exists(REPO):
            subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
        else:
            subprocess.run(["git", "-C", REPO, "pull"], check=True)
        print("✅ Repositorio listo en Drive.")
    except Exception as _e:
        print("⚠️ No se pudo clonar/actualizar el repositorio:", _e)
        print("   Si la carpeta ya existe en tu Drive puedes continuar.")
else:
    print("Modo local: no se monta Drive.")

# ============================================================
# Librerías necesarias
# ============================================================
for _pkg in ("numpy", "scipy", "sympy", "matplotlib", "requests", "ipywidgets", "plotly"):
    try:
        importlib.import_module(_pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

# ============================================================
# Motor de la tarea
# ============================================================
# El motor vive aparte del cuaderno y este recibe solo la API que el motor
# declara en su `__all__`. En Colab se usa el blob; en local, MODO_MOTOR.
sys.path.insert(0, REPO)

if MODO == "modulos":
    from profe.ui.cuaderno import *      # noqa: F401,F403
else:
    if MODO == "blob":
        _ruta = os.path.join(REPO, "grader_ofuscado.txt")
        with open(_ruta, encoding="utf-8") as f:
            _blob = f.read().strip()
        codigo = zlib.decompress(base64.b64decode(_blob)).decode("utf-8")
    else:
        _ruta = os.path.join(REPO, "grader_local.py")
        with open(_ruta, encoding="utf-8") as f:
            codigo = f.read()
    _MOTOR = {}
    try:
        exec(codigo, _MOTOR)
    except Exception as _e:
        raise RuntimeError(
            "No pude cargar el motor desde %s.\n"
            "En local: ejecuta  python tools/build.py  para generarlo.\n"
            "Detalle: %r" % (_ruta, _e))
    globals().update({_n: _MOTOR[_n] for _n in _MOTOR["__all__"]})

print("Motor cargado :", MODO)

# ============================================================
# Preludio estilo MATLAB + robomat
# ============================================================
sys.path.insert(0, os.path.join(REPO, "lib"))
with open(os.path.join(REPO, "lib", "matlab_like.py"), encoding="utf-8") as f:
    exec(f.read(), globals())
from robomat import M, I, sin, cos, tan, var, rad, deg, evaluar, mostrar, jacobiano
print("Preludio MATLAB:", os.path.join(REPO, "lib", "matlab_like.py"))

# ============================================================
# Herramientas de visualización (gráficas interactivas plotly)
# ============================================================
# Viven en tools/visualizar.py, FUERA del motor: no forman parte del sistema
# de calificación, así que puedes leerlas y reutilizarlas.
sys.path.append(os.path.join(REPO, "tools"))
try:
    import visualizar
    importlib.reload(visualizar)
except ImportError as _e:
    raise RuntimeError(
        "No encontré tools/visualizar.py, que las gráficas interactivas usan.\n"
        "En la carpeta de la tarea ese archivo debe estar junto al cuaderno.\n"
        "Detalle: %r" % (_e,))
print("Visualización :", os.path.join(REPO, "tools", "visualizar.py"))

# ============================================================
# Estado del equipo (se guarda en Drive para no perderlo)
# ============================================================
RUTA_ESTADO = os.path.join(REPO, "equipo_u2t4.json")
EQUIPO = None
# El cuaderno 1 guarda aqui el equipo, el correo y el numero de control del jefe;
# asi el cuaderno 2 no tiene que volver a identificarlos.
CORREO = ""
NC_JEFE = ""
_estado = None
if os.path.exists(RUTA_ESTADO):
    try:
        with open(RUTA_ESTADO, encoding="utf-8") as f:
            _estado = json.load(f)
    except Exception as e:
        print("No se pudo leer el estado previo:", e)
if isinstance(_estado, dict) and _estado.get("__estado__") == 1:
    # Formato nuevo (lo escribe este cuaderno): equipo + correo + NC del jefe.
    EQUIPO = _estado.get("equipo")
    CORREO = _estado.get("correo") or ""
    NC_JEFE = _estado.get("nc_jefe") or ""
elif isinstance(_estado, dict):
    # Formato viejo: el equipo a secas. OJO: trae una clave "equipo" con el
    # NUMERO de equipo, por eso la marca de formato es "__estado__" y no "equipo".
    EQUIPO = _estado
if isinstance(EQUIPO, dict):
    # El correo tambien se puede recuperar del integrante jefe (archivos viejos).
    if not CORREO:
        for _i in (EQUIPO.get("integrantes") or []):
            if _i.get("esJefe") and _i.get("correo"):
                CORREO = _i["correo"]
                break
    NC_JEFE = NC_JEFE or EQUIPO.get("jefe", "") or extraer_nc(CORREO) or ""
    print("📄 Estado del equipo recuperado:", EQUIPO.get("tema"), "-", EQUIPO.get("estado"))
else:
    NC_JEFE = NC_JEFE or extraer_nc(CORREO) or ""

import requests

def _post(accion, **campos):
    """Llama a Apps Script y devuelve la carga útil o lanza el error del servidor."""
    payload = {"accion": accion, "token": WEBHOOK_TOKEN}
    payload.update(campos)
    r = requests.post(APPS_SCRIPT_URL, json=payload, timeout=40)
    data = r.json()
    if data.get("status") != "ok":
        raise RuntimeError(data.get("message", "Error desconocido del servidor"))
    return data.get("data", {})

def _guardar_estado():
    # "__estado__" marca el formato: distingue este envoltorio de un archivo
    # viejo, que tambien trae una clave "equipo" (con el numero de equipo).
    with open(RUTA_ESTADO, "w", encoding="utf-8") as f:
        json.dump({"__estado__": 1, "equipo": EQUIPO, "correo": CORREO,
                   "nc_jefe": NC_JEFE}, f, ensure_ascii=False, indent=1)

print("✅ Entorno listo.")


# Guarda el Notebook
Ve a `Archivo → Guardar una copia en Drive` y guárdalo en
`CursoMN/U2_T4_2_Caso` para conservar tu trabajo.


---
# 3. Tu caso de estudio

Aquí abajo aparece **solo tu proyecto** (el catálogo completo está en el `README.md`).

In [ ]:
# @title Muestra tu caso de estudio
from IPython.display import display, Markdown

if not EQUIPO:
    print('⛔ Todavía no tienes tema asignado. Ejecuta primero el cuaderno U2_T4_1_Equipo.')
else:
    TEMA = EQUIPO['tema']
    INFO = CATALOGO[TEMA]
    display(Markdown('## %s · %s' % (TEMA, INFO['titulo'])))
    display(Markdown('**Incógnita:** $%s$  (%s)' % (INFO['incognita'], INFO['unidad'])))
    display(Markdown('**Ecuación del caso:**\n\n$$%s$$' % INFO['ecuacion']))
    display(Markdown('**Desafío principal:** %s' % INFO['desafio']))
    display(Markdown('**Métodos sugeridos:** %s' % ', '.join(INFO['metodos_sugeridos'])))
    _filas = ['| Parámetro | Valor nominal | Incertidumbre |', '|---|---|---|']
    for p, spec in INFO['params'].items():
        if spec[0] == 'u':
            _filas.append('| `%s` | %.4g | uniforme en [%.4g, %.4g] |' % (p, 0.5 * (spec[1] + spec[2]), spec[1], spec[2]))
        else:
            _filas.append('| `%s` | %.6g | ±%.1f%% |' % (p, spec[1], 100 * spec[2]))
    display(Markdown('\n'.join(_filas)))
    if INFO.get('polinomico'):
        display(Markdown('> 🔺 **Tema polinómico:** se exige **Müller o Bairstow** entre tus métodos.'))

---
# 4. Definición simbólica del problema

Con `sympy` el motor construye la función de tu caso y su **derivada** (o el **Jacobiano**,
si es un sistema), y las convierte en funciones rápidas con `lambdify`. Esta celda solo las
**muestra**, para que veas con qué expresión vas a trabajar.

> 📝 **Sobre los nombres.** En el **código** las variables se llaman con letras normales
> (`th1`, `omega`, `eps`), pero **en pantalla** se muestran como notación matemática
> ($\theta_1$, $\omega$, $\varepsilon$). Es el mismo símbolo: en el código usa siempre el
> nombre normal.


In [ ]:
# @title Definición simbólica y funciones numéricas
import numpy as np
import sympy as sp
from IPython.display import display, Markdown

MODELO = modelo(TEMA)          # viene del calificador (oculto)
ES_SISTEMA = MODELO['sistema']

if ES_SISTEMA:
    syms, exprs, sym_params = MODELO['sym']
    F_OFICIAL = exprs
    J_OFICIAL = sp.Matrix(exprs).jacobian(syms)
    display(Markdown('**Sistema de ecuaciones** $F(\\mathbf{x})=\\mathbf{0}$:'))
    for e in exprs:
        display(sp.Eq(e, 0))
    display(Markdown('**Jacobiano** $J=\\partial F/\\partial \\mathbf{x}$:'))
    display(J_OFICIAL)
else:
    x_sym, expr, sym_params = MODELO['sym']
    D_OFICIAL = sp.simplify(sp.diff(expr, x_sym))
    display(Markdown('**Ecuación** $f(%s)=0$:' % sp.latex(x_sym)))
    display(sp.Eq(expr, 0))
    display(Markdown('**Derivada** $f\'$:'))
    display(D_OFICIAL)

# Alias cómodos: ahora puedes escribir th1, omega, x, ... directamente
for _n, _s in zip(MODELO['vars'], (syms if ES_SISTEMA else [x_sym])):
    globals()[_n] = _s
print('Variables disponibles:', ', '.join(MODELO['vars']))
print('Parámetros del caso:', list(MODELO['params']))
print('Valores iniciales de referencia:', MODELO['info'].get('x0', MODELO['info'].get('intervalo')))

In [ ]:
# @title 🔧 Opcional: fija TÚ el intervalo y/o el arranque
#
# El motor simula el error de un analista (por eso algunos métodos fallan en
# algunas muestras). Con estas dos líneas TÚ decides con qué información corren
# y ves de qué dependen los fallos:
#
#   MIS_INTERVALOS -> intervalo (a, b) donde f CAMBIA DE SIGNO. Solo lo usan los
#     métodos que necesitan un PARÉNTESIS (bisección, falsa posición, punto fijo
#     modificado y Brent). Con un intervalo que encierra la raíz, esos métodos
#     NO fallan nunca.
#
#   MIS_ARRANQUES -> punto(s) de arranque de los métodos que NO usan intervalo
#     (abiertos y polinómicos). Debe estar CERCA de la raíz; según el método,
#     los dos números significan:
#       · Newton-Raphson -> usa solo el primero (p0)
#       · Secante        -> usa los dos (p0, p1)
#       · Müller         -> usa los dos y su punto medio como tercero
#       · Bairstow       -> los toma como coeficientes iniciales (r0, s0) de su
#                           factor cuadrático
#     Ojo: si lo pones LEJOS de la raíz, los métodos abiertos fallan MÁS.
#
# Descomenta y ajusta a tu caso (la sección 4 te sugiere los números):

# MIS_INTERVALOS = INFO["intervalo"]   # el intervalo de tu caso
# MIS_ARRANQUES  = (1000, 1500)        # dos valores cerca de la raíz


In [ ]:
# @title 🔍 Aísla la raíz (gráfica interactiva)
# Los métodos que necesitan un PARÉNTESIS (bisección, falsa posición, punto fijo
# modificado, Brent) no pueden ni empezar si en los extremos del intervalo f(x)
# NO cambia de signo. Ese intervalo es el resultado de tu ANÁLISIS GRÁFICO.
import numpy as np
from IPython.display import display, Markdown

try: MIS_LIMITES = MIS_INTERVALOS
except: MIS_LIMITES = None
try: MIS_ARRANQUES_EQUIPO = MIS_ARRANQUES
except: MIS_ARRANQUES_EQUIPO = None

if not EQUIPO:
    print("⛔ Todavía no tienes tema asignado.")
elif ES_SISTEMA:
    print("Tu caso es un SISTEMA de ecuaciones: se resuelve con Newton multivariable")
    print("(Jacobiano), no con métodos de intervalo, así que aquí no hay que aislar.")
else:
    _lo, _hi = INFO["intervalo"]
    _nom = {n: (s[1] if s[0] == "n" else 0.5 * (s[1] + s[2])) for n, s in INFO["params"].items()}
    _raices = oraculo_escalar_todas(lambda z: MODELO["f_arr"](z, _nom), _lo, _hi)
    _r = _raices[0] if _raices else float("nan")

    _xs = np.linspace(_lo, _hi, 800)
    _fig = visualizar.graficar_funcion(
        lambda z: MODELO["f_arr"](np.asarray(z, dtype=float), _nom),
        _xs, raiz=_r, nombre_var=MODELO["vars"][0],
        titulo="Tu caso: %s — %s" % (TEMA, INFO["titulo"]))
    _fig.show(config=visualizar.config_plotly())

    _fa = MODELO["f_arr"](np.array([_lo]), _nom)[0]
    _fb = MODELO["f_arr"](np.array([_hi]), _nom)[0]
    print("Intervalo del tema : (%.6g, %.6g)" % (_lo, _hi))
    print("f(extremo izquierdo) = %+.6g" % _fa)
    print("f(extremo derecho)   = %+.6g" % _fb)
    print("¿Cambia de signo?    : %s" % ("SÍ ✅ el intervalo aísla la raíz"
                                          if _fa * _fb < 0 else "NO ⚠️"))
    print("Raíz con valores nominales: %.8g" % _r)
    print("Número de cruces del eje dentro del intervalo: %d" % len(_raices))

    display(Markdown(
        "La gráfica se dibuja sobre el **intervalo del tema**, (%g, %g): ése es el rango de "
        "búsqueda del caso, no un dato que cambie.  \n"
        "Verifica que la curva cruce el eje **una sola vez** ahí: ese cruce es la **raíz nominal**."
        % (_lo, _hi)))

    if MIS_LIMITES is None:
        display(Markdown(
            "> ℹ️ **No fijaste `MIS_INTERVALOS`.** La simulación arranca con el error de "
            "*aislamiento* que cometería un analista: el intervalo que propone **no siempre** "
            "encierra la raíz. Por eso los métodos de paréntesis fallan en algunas muestras — "
            "**no porque diverjan**, sino porque no pueden ni empezar."))
    else:
        _a, _b = MIS_LIMITES
        _za = MODELO["f_arr"](np.array([_a]), _nom)[0]
        _zb = MODELO["f_arr"](np.array([_b]), _nom)[0]
        _ok = _za * _zb < 0
        print("\nTu intervalo (%g, %g): f(a)=%+.4g  f(b)=%+.4g  →  %s"
              % (_a, _b, _za, _zb, "AÍSLA ✅" if _ok else "NO aísla ❌"))
        if _ok:
            print("   f(a) y f(b) tienen signo opuesto: encierra la raíz NOMINAL (ancho %.4g)." % (_b - _a))
            display(Markdown(
                "> ✅ Encierra la raíz **nominal**, que es lo necesario para que un método de "
                "paréntesis pueda empezar.  \n"
                "> **Pero no es suficiente**: la incertidumbre del caso mueve la raíz, así que el "
                "intervalo debe cubrir **todo el rango** de abajo. Si se queda corto, esos métodos "
                "fallarán precisamente en las muestras cuya raíz caiga fuera."))
        else:
            display(Markdown(
                "> ⚠️ Tu intervalo **no** aísla la raíz, así que los métodos de paréntesis "
                "fallarán en la simulación. Busca dos valores donde la curva cambie de signo."))

# --- ¿Qué intervalo debe ENCERRAR la raíz? (ojo: la raíz NO es un punto fijo) ---
if EQUIPO and not ES_SISTEMA and np.isfinite(_r):
    print("\nIntervalo del TEMA (el que dibuja la gráfica): (%g, %g)" % (_lo, _hi))
    _muestras = muestrear(TEMA, N=200, seed=semilla(TEMA, NC_JEFE or "0"))
    _rango = []
    for _i in range(200):
        _v = {_p: _muestras[_p][_i] for _p in MODELO["params"]}
        _cand = oraculo_escalar_todas(lambda _z: MODELO["f_arr"](_z, _v), _lo, _hi)
        if _cand:
            _rango.append(min(_cand, key=lambda _z: abs(_z - _r)))
    if _rango:
        _rmin, _rmax = min(_rango), max(_rango)
        print("Raíz nominal = %.6g   |   en 200 escenarios la raíz cae en [%.6g, %.6g]"
              % (_r, _rmin, _rmax))
        print("La incertidumbre del caso MUEVE la raíz %.4g de ancho (y en otras muestras"
              % (_rmax - _rmin))
        print("puede salir aún más lejos): tu MIS_INTERVALOS debe cubrir TODO ese rango,")
        print("no sólo el cruce que ves en la gráfica.")
        print("\nCopia esto en la celda de arriba y vuelve a correr la sección 6:")
        print("   MIS_INTERVALOS = (%g, %g)" % (_lo, _hi))
        print("   MIS_ARRANQUES  = (%.6g, %.6g)   # abiertos y polinómicos" % (0.95 * _r, 1.05 * _r))
        print("Los de paréntesis usan el intervalo y los abiertos el arranque: ninguno falla.")
        if MIS_LIMITES is not None and not (_a <= _rmin and _b >= _rmax):
            print("\n⚠️ Tu MIS_INTERVALOS (%g, %g) encierra la raíz nominal, pero NO cubre" % (_a, _b))
            print("   el rango [%.6g, %.6g]: fallará en las muestras cuya raíz quede fuera." % (_rmin, _rmax))
            print("   Para ver el 100%% usa (%g, %g)." % (_lo, _hi))


---
# 5. Elige los métodos que vas a comparar

Esta tarea es **teórica**: los métodos **ya vienen implementados** en el motor, así que no
programas nada. Tu trabajo es **elegir cuáles comparar** y **leer las gráficas**.

En las casillas de abajo **marca**:

1. **todos** los métodos **sugeridos** de tu caso (los viste en la sección 3), y
2. **al menos un contraejemplo**: un método que **no** esté sugerido, para contrastar en tu
   informe por qué en tu caso rinde distinto.

Las casillas están agrupadas **por familia** (cerrados, abiertos, híbrido, polinomios) para que
veas de inmediato qué estás comparando. El recuadro de la derecha te dice **en vivo** si tu
selección cumple la regla. En los casos de **sistema** (T14, T15, T19) no se pide contraejemplo:
sus dos métodos ya son los sugeridos.


In [ ]:
# @title 🎛️ Elige los métodos (marca las casillas) — esto es lo que se autocalifica
# Los métodos YA vienen implementados en el motor: esta tarea es teórica, así
# que no programas nada. Aquí solo eliges CUÁLES comparar.
#
# Marca las casillas:  TODOS los métodos sugeridos de tu caso  +  al menos UN
# método que NO esté sugerido (un contraejemplo, para contrastar en tu informe).
# En los casos de SISTEMA no se pide contraejemplo: sus dos métodos ya son los
# sugeridos.
import ipywidgets as widgets
from IPython.display import display

_DISPONIBLES = mapa_metodos(es_sistema(TEMA))    # {nombre: método} (lo pone el motor)
_SUGERIDOS = metodos_sugeridos_para(TEMA)

# Una casilla por método, agrupadas por familia para que se lean de un vistazo.
_FAMILIAS_UI = [("cerrado", "Cerrados (intervalo)"), ("abierto", "Abiertos"),
                ("hibrido", "Híbrido"), ("polinomios", "Polinomios"),
                ("desconocida", "Otros")]

CASILLAS = {}
_columnas = []
for _clave, _titulo in _FAMILIAS_UI:
    _miembros = [n for n in _DISPONIBLES if familia(n) == _clave]
    if not _miembros:
        continue
    _cajas = [widgets.HTML("<b>%s</b>" % _titulo)]
    for _n in _miembros:
        _cajas.append(widgets.Checkbox(value=(_n in _SUGERIDOS), description=_n,
                                       indent=False,
                                       layout=widgets.Layout(width="290px")))
        CASILLAS[_n] = _cajas[-1]
    _columnas.append(widgets.VBox(_cajas))

BT_SUGERIDOS = widgets.Button(description="Marcar solo los sugeridos")
OUT_METODOS = widgets.Output()
METODOS = {}

def _aplicar(*_):
    """METODOS = solo los métodos cuyas casillas están marcadas."""
    global METODOS
    METODOS = {n: _DISPONIBLES[n] for n in _DISPONIBLES if CASILLAS[n].value}
    with OUT_METODOS:
        OUT_METODOS.clear_output()
        if not METODOS:
            print("⚠️ No has marcado ningún método.")
            return
        for _n in METODOS:
            print("   %s %-28s familia: %s"
                  % ("⭐" if _n in _SUGERIDOS else "·", _n, familia(_n)))
        _faltan = [m for m in _SUGERIDOS if m not in METODOS]
        _extra = [n for n in METODOS if n not in _SUGERIDOS]
        print()
        if _faltan:
            print("⚠️ Te faltan métodos sugeridos: " + ", ".join(_faltan))
        else:
            print("✅ Incluyes todos los métodos sugeridos.")
        if _extra:
            print("· Contraejemplos (no sugeridos): " + ", ".join(_extra))
        elif not es_sistema(TEMA):
            print("⚠️ Agrega al menos un método NO sugerido (un contraejemplo).")

def _marcar_sugeridos(_):
    for _n, _cas in CASILLAS.items():
        _cas.value = (_n in _SUGERIDOS)

for _cas in CASILLAS.values():
    _cas.observe(_aplicar, names="value")
BT_SUGERIDOS.on_click(_marcar_sugeridos)

display(widgets.HBox(_columnas, layout=widgets.Layout(grid_gap="16px")))
display(BT_SUGERIDOS)
display(OUT_METODOS)
_aplicar()


---
# 6. Simulación de Monte Carlo

Se perturban los parámetros del caso (incertidumbre real) y se corre **cada método** sobre
las **mismas** $N$ muestras. Después se compara cada resultado contra la **raíz de
referencia** (el *oráculo*), muestra por muestra.

> ### ¿Qué es la "incertidumbre" aquí?
> Son las **tolerancias de los componentes y el ruido de los sensores**: en cada muestra se
> cambian un poco los **parámetros del caso** (el catálogo de la sección 3 lista cuáles y con
> qué tolerancia) alrededor de su valor nominal, y se recalcula la raíz. **No** es el valor
> inicial.

> ### ¿Y el punto de arranque?
> También cambia entre muestras, pero por **otra** razón: se modela el **error al aislar la
> raíz** después de ver la gráfica (±15 % del ancho del intervalo en la mayoría de los casos, y
> bastante peor en el resto). Eso es lo que hace que un método abierto a veces falle: su
> arranque es realista, no perfecto.


In [ ]:
# @title Ejecutar Monte Carlo
if not METODOS:
    raise ValueError("Primero elige al menos un método con el selector (sección 5).")

N_MC = 1000                              # tamaño de la simulación
SEMILLA_EQUIPO = semilla(TEMA, NC_JEFE)  # misma semilla para todos los métodos
_INTERVALO = globals().get("MIS_LIMITES", None)
_ARRANQUES = globals().get("MIS_ARRANQUES", None)

print("Simulando %d muestras para el tema %s..." % (N_MC, TEMA))
if _INTERVALO is not None:
    print("Usando TU intervalo %s para los métodos que necesitan paréntesis." % (_INTERVALO,))
if _ARRANQUES is not None:
    print("Usando TU arranque %s para los métodos que NO usan intervalo (abiertos y polinómicos)."
          % (_ARRANQUES,))
try:
    RES = evaluar_tema(TEMA, METODOS, N=N_MC, seed=SEMILLA_EQUIPO,
                       intervalo=_INTERVALO, arranques=_ARRANQUES)
except ValueError as _e:
    raise ValueError("Revisa MIS_ARRANQUES en la sección 4: %s" % _e)
print("Listo.\n")

print("%-30s %8s %8s %10s %10s %10s" % ("Método", "%éxito", "%diverg", "mediana", "media", "mediana_err"))
print("-" * 84)
for _m, _d in RES["metodos"].items():
    print("%-30s %7.1f%% %7.1f%% %10.0f %10.1f %10.1e" % (
        _m, 100 * _d["pct_exito"], 100 * _d["pct_diverge"],
        np.nanmedian(_d["n_iter"]), np.nanmean(_d["n_iter"]),
        np.nanmedian(_d["err_ref"])))
print("\nUn método \"CUENTA\" si acierta la raíz de referencia en >= %.0f%% de las muestras."
      % (100 * UMBRAL_EXITO))

# --- ¿POR QUÉ falló cada método? No todo es "divergencia". ---
print("\n" + "=" * 84)
print("¿POR QUÉ FALLÓ CADA MÉTODO?")
print("=" * 84)
for _m, _d in RES["metodos"].items():
    _fallos = {k: v for k, v in _d.get("motivos", {}).items() if k != "ok"}
    if not _fallos:
        print("   %-28s sin fallos ✅" % _m)
        continue
    print("   %-28s (%d de %d muestras)" % (_m, _d["diverge"], _d["n_iter"].size))
    for _k, _v in sorted(_fallos.items(), key=lambda kv: -kv[1]):
        print("        %5d   %s" % (_v, MOTIVOS_TEXTO.get(_k, _k)))
print("\nOjo: \"no arrancó\" NO es que el método diverja: es que el intervalo no encerraba la raíz.")
if _INTERVALO is None:
    print("Prueba fijando MIS_INTERVALOS (sección 4) con un intervalo válido y vuelve a correr: esos métodos dejan de fallar.")
if _ARRANQUES is None:
    print("Prueba fijando MIS_ARRANQUES (sección 4) con un arranque CERCA de la raíz y vuelve a correr: los métodos abiertos dejan de fallar.")


---
# 7. Gráficos de desempeño

Primero el **histograma comparativo** y la **tasa de divergencia**; luego el
**box plot** (la vista más informativa) y la **distribución de la respuesta física**.

> 🖱️ Las gráficas son **interactivas**: acerca con la rueda, **selecciona puntos** con la caja
> o el lazo, y descarga la imagen con el icono de cámara.


In [ ]:
# @title 📊 Histograma de iteraciones y % de divergencia (interactivo)
_fig = visualizar.graficar_histograma(RES)
_fig.show(config=visualizar.config_plotly())


In [ ]:
# @title ⭐ Box plot comparativo de iteraciones (interactivo)
_fig = visualizar.graficar_boxplot(RES)
_fig.show(config=visualizar.config_plotly())

_met = list(RES["metodos"].keys())
_ord = sorted(_met, key=lambda m: np.nanmedian(RES["metodos"][m]["n_iter"]))
print("Ordenados por mediana de iteraciones (más rápido primero):")
for m in _ord:
    d = RES["metodos"][m]
    print("   %-30s mediana=%4.0f  dispersión(IQR)=%4.0f  fallos=%5.1f%%" % (
        m, np.nanmedian(d["n_iter"]),
        np.nanpercentile(d["n_iter"], 75) - np.nanpercentile(d["n_iter"], 25),
        100 * d["pct_diverge"]))


### Distribución de la respuesta física

Este gráfico **no** compara métodos: muestra **cómo se reparte la respuesta del sistema**
cuando los componentes varían.

- **Cada muestra** tiene sus propios parámetros (ligeramente distintos a los nominales) y se
  obtiene **su** raíz verdadera con el oráculo. Aquí se grafica el conjunto de esas raíces.
- El **eje horizontal** es la **respuesta física** de tu caso (la temperatura $T$, el ángulo
  $\theta$, el volumen $v$, …), **no** las iteraciones ni una comparación de métodos.
- El **ancho** del histograma mide la **sensibilidad**: si es angosto, el sistema es **robusto**
  (la tolerancia de los componentes casi no mueve la respuesta); si es ancho, la respuesta
  **depende mucho** de ellos y conviene apretar sus tolerancias.
- Debajo se reporta **media, mediana, desviación, mínimo y máximo**. La **desviación** frente al
  valor nominal es, en la práctica, la incertidumbre **de salida**.

> 🔍 **Para el informe:** ¿qué parámetro de tu caso mueve más la respuesta? No todos pesan
> igual, porque cada uno tiene distinta tolerancia.

> 🔹 En los casos de **sistema** (T14, T15, T19) se grafica un histograma **por incógnita**.


In [ ]:
# @title 📈 Distribución de la respuesta física ante la incertidumbre (interactivo)
_fig = visualizar.graficar_distribucion(RES, modelo=MODELO, es_sistema=ES_SISTEMA)
_fig.show(config=visualizar.config_plotly())

if not ES_SISTEMA:
    _rr = RES["raiz_ref"][np.isfinite(RES["raiz_ref"])]
    print("Respuesta física: media=%.6g  mediana=%.6g  desv=%.3g  min=%.6g  max=%.6g"
          % (np.mean(_rr), np.median(_rr), np.std(_rr), np.min(_rr), np.max(_rr)))

---
# 8. Conclusiones de ingeniería

El notebook calcula dos respuestas a partir de **tus propios resultados** y te recuerda las
tres preguntas conceptuales. Todas se entregan en el **informe**; el profesor las revisa.

> 🎓 **La pregunta de la exposición:** *"Si estuvieran diseñando el firmware del controlador
> de este sistema mecatrónico en tiempo real, ¿qué método programarían y por qué?"*


In [ ]:
# @title 🎓 Preguntas de criterio (para el informe — NO se autocalifican)
from IPython.display import display, Markdown

_conteo = [m for m, d in RES["metodos"].items() if d["cuenta"]] or list(RES["metodos"].keys())
_rapido = min(_conteo, key=lambda m: np.nanmedian(RES["metodos"][m]["n_iter"]))
_fallador = max(RES["metodos"].keys(), key=lambda m: RES["metodos"][m]["pct_diverge"])

display(Markdown("### 1. Según TU simulación, ¿qué método necesitó MENOS iteraciones?"))
display(Markdown("**%s** (mediana %d iteraciones)." % (
    _rapido, np.nanmedian(RES["metodos"][_rapido]["n_iter"]))))

display(Markdown("### 2. ¿Qué método falló (no llegó a la raíz) en MÁS muestras?"))
display(Markdown("**%s** (%.1f%% de divergencia)." % (
    _fallador, 100 * RES["metodos"][_fallador]["pct_diverge"])))

display(Markdown("### 3. Preguntas conceptuales (para la exposición)"))
for _p in PREGUNTAS_CONCEPTO:
    display(Markdown("- **%s**: %s" % (_p["id"], _p["texto"])))

display(Markdown(
    "> 🎓 **La pregunta de la exposición:** *\"Si estuvieran diseñando el firmware del "
    "controlador de este sistema mecatrónico en tiempo real, ¿qué método programarían y por qué?\"*"))


---
# 9. Calificación automática y envío

Lo **único** que se califica solo es la **selección de métodos**: que hayas incluido los
**sugeridos** de tu caso y al menos un **contraejemplo**. El resto (informe, interpretación de
las gráficas y exposición) lo evalúa el **profesor a mano**.

> ⛔ **No se envía si la selección está incompleta.** La celda de abajo revisa primero y, si
> te falta algún método, **te dice cuál** y no manda nada (así no gastas un intento en balde).
> Corrige las casillas de la **sección 5** y vuelve a ejecutarla.


In [ ]:
# @title ✅ Calificar selección y enviar
NOTA_METODOS, DESGLOSE = calificar(TEMA, list(METODOS.keys()), RES)

print("=" * 74)
print("AUTOCALIFICACIÓN — SELECCIÓN DE MÉTODOS")
print("=" * 74)
print("Caso: %s — %s" % (TEMA, RES["titulo"]))
print("Seleccionaste  :", ", ".join(DESGLOSE["seleccion"]))
print("Sugeridos      :", ", ".join(DESGLOSE["sugeridos"]))
print("Faltantes      :", ", ".join(DESGLOSE["faltantes"]) if DESGLOSE["faltantes"] else "(ninguno) ✅")
print("Contraejemplos :", ", ".join(DESGLOSE["contraejemplos"]) if DESGLOSE["contraejemplos"]
      else ("(no se pide en casos de sistema)" if not DESGLOSE["requiere_contraejemplo"] else "(ninguno) ⚠️"))
print("Familias       :", ", ".join(DESGLOSE["familias"]))

if DESGLOSE["aprueba"]:
    print("\n✅ SELECCIÓN CORRECTA: incluiste los métodos correctos"
          + ("" if not DESGLOSE["requiere_contraejemplo"] else " y al menos un contraejemplo") + ".")

print("\n💡 Consejos:")
for _c in consejos(RES):
    print("   · %s" % _c)

# --- Envío: SOLO si la selección cumple la regla ---
print()
if not DESGLOSE["aprueba"]:
    print("=" * 74)
    print("⛔ NO SE ENVIÓ: tu selección todavía no cumple la regla.")
    print("=" * 74)
    if DESGLOSE["faltantes"]:
        print("   · Te faltan métodos SUGERIDOS:", ", ".join(DESGLOSE["faltantes"]))
    if DESGLOSE["requiere_contraejemplo"] and not DESGLOSE["contraejemplos"]:
        print("   · Te falta al menos un método NO sugerido (un contraejemplo).")
    print("\n   Ve a la sección 5, corrige las casillas y ejecuta esta celda otra vez.")
    print("   No se gastó ningún intento.")
elif not EQUIPO:
    print("⛔ No se puede enviar: no hay equipo registrado.")
else:
    try:
        _r = enviar_resultado(NC_JEFE, TEMA, list(METODOS.keys()), NOTA_METODOS,
                              correo=CORREO, alcance="equipo")
        print("✅ Enviado a la hoja de resultados:", _r.get("data", _r).get("mensaje", "ok"))
        print("   Equipo %s · tema %s · %s integrante(s) actualizados"
              % (EQUIPO.get("equipo"), TEMA, _r.get("data", _r).get("actualizados", 1)))
        print("\n   Puedes comprobarlo con la celda de ABAJO (no gasta intentos).")
    except Exception as e:
        print("⚠️ No se pudo enviar todavía:", e)
        print("   Guarda el notebook y avisa al profesor. Tu resultado NO se perdió.")

print("\n--- Guarda el notebook en Drive (Archivo → Guardar) ---")


---
# 10. Comprobar lo que ya está guardado

Puedes consultar **en cualquier momento** qué quedó registrado en la hoja de resultados: tu
tema, el estado del equipo y —integrante por integrante— la **calificación** y los **métodos**
que envió cada quien.

> 🔹 Esta consulta **no envía nada**, **no gasta intentos** y **no modifica** nada: solo lee.


In [ ]:
# @title 🔎 Comprobar lo que ya está guardado (no envía nada)
if not EQUIPO:
    print("⛔ No hay equipo registrado. Ejecuta primero el cuaderno U2_T4_1_Equipo.")
else:
    try:
        _eq = _post("proyecto_equipo", NC=NC_JEFE, correo=CORREO).get("equipo")
    except Exception as e:
        print("⚠️ No pude consultar el servidor:", e)
        _eq = None

    if not _eq:
        print("No encontré tu equipo en el servidor (revisa tu conexión).")
    else:
        print("Equipo %s · tema %s — %s" % (_eq.get("equipo"), _eq.get("tema"), _eq.get("titulo")))
        print("Estado del tema: %s\n" % _eq.get("estado"))
        print("%-11s %-32s %13s   %s" % ("NC", "Nombre", "Calificación", "Métodos enviados"))
        print("-" * 100)
        _pendientes = 0
        for _i in _eq.get("integrantes", []):
            _cal = _i.get("calificacion")
            if _cal is None:
                _pendientes += 1
            print("%-11s %-32s %13s   %s" % (
                _i.get("nc"), (_i.get("nombre") or "")[:32],
                ("sin enviar" if _cal is None else "%.2f" % _cal),
                _i.get("metodos") or "—"))
        print()
        if _pendientes:
            print("⏳ Faltan %d integrante(s) por enviar." % _pendientes)
            print("   Si la selección ya cumple la regla, usa la celda de arriba para enviar.")
        else:
            print("✅ Todo el equipo ya envió.")
        print("\n(Esta consulta no gasta intentos ni modifica nada.)")
